# 🎯 06 — Agent, Handoff and End-to-End Evaluation (Levels 2 and 3)

Levels 2 and 3. Routing as a classification problem with a confusion matrix, handoffs as a
question of what survived the hop, trajectories as four different definitions of "the right
path", and the end-to-end check no framework will do for you: **the state of the database**.

## Learning Objectives
In this notebook, you will learn:
1. **Routing accuracy and its confusion matrix** - which intents get mistaken for which, plus a confidence-threshold sweep
2. **Handoff integrity** - did the customer id, the issue and the prior findings survive the hop between agents?
3. **Four trajectory matching modes** - exact, in-order, any-order, contains-required, as short functions
4. **End-to-end** - `TaskCompletionMetric`, `StepEfficiencyMetric`, RAGAS `AgentGoalAccuracy`, and a SQLite final-state check
5. **Blame attribution** - walking a level-3 failure back down to the level that caused it

## Prerequisites
- Notebooks 02, 03 and 05 (`data/lumen.db`, `data/golden_set.json`, `data/trajectories.json`)

---
## 🗺️ The concept

```
  LEVEL 1   tool call        did get_invoice fire with customer_id=CUS-101?
     |
  LEVEL 2   agent/handoff    did supervisor route to billing?
     |                       did prior_findings survive knowledge -> billing?
     |
  LEVEL 3   end-to-end       did the customer get their money back?
     |
     +--> SQLite:  SELECT COUNT(*) FROM refunds WHERE invoice_id='INV-2041'
                   expected: exactly 1, amount 49.00, approved_by NOT NULL

  Blame attribution walks this UPWARDS: level 3 failed, so which of
  levels 1 and 2 failed first? That is the difference between "the agent
  is bad" and "the supervisor misroutes Spanish billing questions".
```

### Key Concepts:
- **Routing is classification.** It has a confusion matrix, and the off-diagonal cells are
  where the product bugs live. "87% accurate" hides that every Spanish billing question goes
  to the account agent.
- **A trajectory has no single correct shape.** Four defensible definitions of "right path"
  disagree on the same run, and choosing between them is a product decision, not a technical one.
- **Task completion is not a transcript property.** The agent can describe a refund it never
  issued. Only the database settles it.

---
## 🔨 Build it

### 0.1 ⚙️ Setup

This notebook picks up the trajectories notebook 05 wrote, rather than re-running the tool
layer. That is the course's handoff style: **a file, not an import**.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: golden set, notebook 05's trajectories, caches
# ============================================================================
import os
import json
import hashlib
import concurrent.futures
import sqlite3
import asyncio
import threading
import warnings
import statistics
from collections import defaultdict

from dotenv import load_dotenv

warnings.filterwarnings("ignore")
# DeepEval reports anonymous usage on the first metric it builds. That call is
# not on the critical path of an evaluation, but it happens inside it — so a
# slow one becomes a slow metric. Off, here and in CI.
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# DeepEval also wraps every model call in a timeout of its own, implemented as
# one worker thread per call. That wrapper is what stalls in this course's
# environment — the request underneath it is fast — so it is switched off. The
# OpenAI client keeps its own request timeout, and judge_bounded() below retries
# a call that really does fail.
os.environ.setdefault("DEEPEVAL_DISABLE_TIMEOUTS", "1")

load_dotenv(os.path.abspath(os.path.join("..", "..", ".env")))

DATA_DIR = os.path.abspath("data")
SCORE_CACHE = os.path.join(DATA_DIR, "score_cache")
DB_PATH = os.path.join(DATA_DIR, "lumen.db")
os.makedirs(SCORE_CACHE, exist_ok=True)

with open(os.path.join(DATA_DIR, "golden_set.json"), encoding="utf-8") as f:
    GOLDEN = json.load(f)

with open(os.path.join(DATA_DIR, "trajectories.json"), encoding="utf-8") as f:
    TRAJECTORIES = json.load(f)

CASES = GOLDEN["cases"]
CASE_BY_ID = {c["case_id"]: c for c in CASES}

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "sk-placeholder-deterministic-metrics-only"

print("✅ Setup complete")
print("🗂️  golden set v%s — %d cases" % (GOLDEN["datasheet"]["version"], len(CASES)))
print("🧭 trajectories from notebook 05:", len(TRAJECTORIES))

✅ Setup complete
🗂️  golden set v1.0.0 — 40 cases
🧭 trajectories from notebook 05: 25


In [2]:
# ============================================================================
# FAULT SWITCHES: the three that break levels 2 and 3
# ============================================================================
FAULTS = {
    "wrong_route": False,           # supervisor sends billing questions to account
    "lost_handoff_context": False,  # the second agent forgets what the first found
    "retry_loop": False,            # the same tool, over and over, to the same end
}

print("🎛️  Faults armed:", {k: v for k, v in FAULTS.items() if v} or "none (clean run)")

🎛️  Faults armed: none (clean run)


In [3]:
# ============================================================================
# SCORE CACHE: identical scheme to notebooks 04 and 05
# ============================================================================
SCORE_MODE = "auto"        # "auto"   = replay if cached, judge + cache on a miss
                           # "strict" = raise on a miss (use in CI, never spends)
                           # "record" = re-judge everything, ignoring the cache


def score_key(metric_name, payload):
    blob = json.dumps({"metric": metric_name, "payload": payload},
                      sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]


def _cache_path(metric_name, payload):
    return os.path.join(SCORE_CACHE, score_key(metric_name, payload) + ".json")


def _read_cache(path):
    if SCORE_MODE != "record" and os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)["score"]
    return None


def _miss(metric_name, path):
    """What to do when nothing is cached. `strict` refuses to spend money."""
    if SCORE_MODE == "strict":
        raise FileNotFoundError(
            "No cached score for " + metric_name + " (" + os.path.basename(path) + ").\n"
            "SCORE_MODE is 'strict', so nothing will be judged. Run the notebook "
            "with SCORE_MODE = 'auto' to fill the cache.")


def _write_cache(path, metric_name, payload, score):
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"metric": metric_name, "payload": payload, "score": score},
                  f, ensure_ascii=False, indent=2)
    return score


# The model that JUDGES, which is not the model the agent RUNS on. A judge is
# picked for reliability first and price second: gpt-4o-mini occasionally
# degenerates into a repetition loop under a strict JSON schema and generates
# until its token cap, which arrives as a 90-second stall rather than an error.
JUDGE_MODEL = "gpt-4.1-mini"

# Judge one case at a time. DeepEval's async path runs its own event loop per
# call; over a long sequential run that path can stall with the request in
# flight and no timeout firing. Nothing here wants concurrency anyway — the
# loops below are sequential so the printed table stays in case order.
JUDGE_ASYNC = False

JUDGE_TIMEOUT_SECONDS = 60   # a judge call slower than this is stuck, not slow
JUDGE_ATTEMPTS = 3           # ...and a stuck call is worth asking again


def judge_bounded(compute):
    """Call the judge under a hard ceiling, and retry a call that misses it.

    Judges are LLM calls and fail the way LLM calls fail; the worst of those
    failures is the one that never returns. Bounding it is the harness's job —
    "the suite never finished" is the least useful result CI can give you.
    A worker thread is safe here because every metric is built with
    async_mode=False, so nothing touches the notebook kernel's event loop.
    """
    for attempt in range(JUDGE_ATTEMPTS):
        pool = concurrent.futures.ThreadPoolExecutor(max_workers=1)
        try:
            return pool.submit(compute).result(timeout=JUDGE_TIMEOUT_SECONDS)
        except Exception as error:
            print("⚠️  judge call failed (%s) — attempt %d of %d"
                  % (type(error).__name__, attempt + 1, JUDGE_ATTEMPTS))
        finally:
            pool.shutdown(wait=False)   # a stuck worker is a daemon; let it go
    raise RuntimeError(
        "A judge call failed %d times in a row. Nothing was cached, so no case "
        "silently became a 0.0 — re-run the cell once the judge is healthy."
        % JUDGE_ATTEMPTS)


def run_async(coro):
    """Run a coroutine from ordinary synchronous code, notebook or not.

    RAGAS metrics are async-only, and a Jupyter kernel is already running an
    event loop of its own — which is why asyncio.run() alone raises "cannot be
    called from a running event loop" here. Running the coroutine on its own
    loop, in its own thread, works in both places.
    """
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)      # no loop in this thread: the plain path

    box = {}

    def target():
        box["value"] = asyncio.run(coro)

    thread = threading.Thread(target=target)
    thread.start()
    thread.join()
    return box["value"]

def cached_measure(metric_name, payload, compute):
    """Judge once, reuse forever. A cached verdict is what lets notebooks 10 and
    11 compare runs — a score that moves while nothing moved is not a measurement.
    """
    path = _cache_path(metric_name, payload)
    hit = _read_cache(path)
    if hit is not None:
        return hit
    _miss(metric_name, path)
    return _write_cache(path, metric_name, payload, judge_bounded(compute))

print("💾 SCORE_MODE =", SCORE_MODE)

💾 SCORE_MODE = auto


---
## 🧭 Part 1: Routing is a classification problem

The supervisor picks one of four routes. That makes it a four-class classifier, and it should
be measured like one — **accuracy tells you almost nothing; the confusion matrix tells you the
bug**.

An agent that is 87% accurate overall might be 100% on English and 40% on Spanish, sending
every Spanish billing question to the account agent. The average is fine. The product is broken
for a fifth of your customers.

In [4]:
# ============================================================================
# SUPERVISOR: notebook 02's router, redefined here and called live
# ============================================================================
ROUTES = ["knowledge", "billing", "account", "escalation"]

SUPERVISOR_PROMPT = """You route customer messages to one specialist.

knowledge  - questions about policy or documentation ("what is your refund window?")
billing    - invoices, charges, refunds, payment problems
account    - plans, passwords, team members, account settings
escalation - anger, legal threats, repeated failures, anything needing a human

Reply with exactly one word from: knowledge, billing, account, escalation.
If the message needs both documentation AND an action, choose the ACTION route -
the specialist can consult the knowledge agent itself."""


_llm = None


def generate(messages):
    """The router's live call. Only JUDGE verdicts are cached in this course."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    return _llm.invoke(messages).content.strip(), 0


def supervise(user_input):
    """Return (route, confidence). Confidence is the router's own hedging, made
    visible: a one-word answer is confident, a hedged sentence is not."""
    text, _ = generate([
        {"role": "system", "content": SUPERVISOR_PROMPT},
        {"role": "user", "content": user_input},
    ])
    words = text.strip().lower().split()
    route = words[0] if words else "escalation"
    if route not in ROUTES:
        route = "escalation"          # unparseable => a human looks at it

    # A crude but honest confidence signal: the fewer extra words the router
    # produced, the more decisive it was. Replace with logprobs if you have them.
    confidence = 1.0 if len(words) == 1 else max(0.3, 1.0 - 0.1 * (len(words) - 1))

    if FAULTS["wrong_route"] and route == "billing":
        route = "account"             # the fault: one specific confusion
    return route, confidence


print("✅ supervise() ready — live calls, one per case")

✅ supervise() ready — live calls, one per case


In [5]:
# ============================================================================
# RUN THE ROUTER over every golden case
# ============================================================================

def route_all(cases):
    rows = []
    for c in cases:
        route, confidence = supervise(c["input"])
        rows.append({
            "case_id": c["case_id"],
            "expected": c["expected_route"],
            "actual": route,
            "confidence": confidence,
            "strata": c["strata"],
        })
    return rows


ROUTING = route_all(CASES)
correct = sum(1 for r in ROUTING if r["actual"] == r["expected"])

print("🧭 routing accuracy: %.1f%% (%d of %d)"
      % (100.0 * correct / len(ROUTING), correct, len(ROUTING)))

🧭 routing accuracy: 85.0% (34 of 40)


In [6]:
# ============================================================================
# CONFUSION MATRIX: the number that actually names the bug
# ============================================================================
# Rows are what the case NEEDED, columns are what the supervisor CHOSE.
# Everything off the diagonal is a product bug with a name.

def confusion_matrix(rows):
    matrix = defaultdict(lambda: defaultdict(int))
    for r in rows:
        matrix[r["expected"]][r["actual"]] += 1
    return matrix


def print_confusion(rows, title="all cases"):
    matrix = confusion_matrix(rows)
    print("── confusion matrix: %s ──" % title)
    print("%-12s %s" % ("expected \\ got", " ".join("%11s" % r for r in ROUTES)))
    for expected in ROUTES:
        cells = " ".join("%11d" % matrix[expected][got] for got in ROUTES)
        total = sum(matrix[expected].values())
        hit = matrix[expected][expected]
        rate = ("%.0f%%" % (100.0 * hit / total)) if total else "  -"
        print("%-12s %s   (%s)" % (expected, cells, rate))
    print()


print_confusion(ROUTING)

# The same matrix, per language — this is where an average hides a segment.
for lang in sorted({r["strata"]["lang"] for r in ROUTING}):
    subset = [r for r in ROUTING if r["strata"]["lang"] == lang]
    hits = sum(1 for r in subset if r["actual"] == r["expected"])
    print("🌐 %s: %.0f%% (%d cases)" % (lang, 100.0 * hits / len(subset), len(subset)))

── confusion matrix: all cases ──
expected \ got   knowledge     billing     account  escalation
knowledge              8           4           1           1   (57%)
billing                0          13           0           0   (100%)
account                0           0           8           0   (100%)
escalation             0           0           0           5   (100%)

🌐 en: 86% (22 cases)
🌐 es: 80% (10 cases)
🌐 hi: 88% (8 cases)


### 1.1 🎚️ The confidence-threshold sweep

If the router reports a confidence, you can refuse to act on the low-confidence ones and send
them to a human instead. That trades **coverage** (how many cases the agent handles alone) for
**precision** (how often it is right when it does act).

The sweep below is a shadow run: it changes no behaviour, it only asks *"what would have
happened at each threshold?"*. That is the cheapest way to pick one.

In [7]:
# ============================================================================
# THRESHOLD SWEEP: coverage vs precision, computed on runs you already have
# ============================================================================

def sweep(rows, thresholds):
    out = []
    for t in thresholds:
        acted = [r for r in rows if r["confidence"] >= t]
        if not acted:
            out.append({"threshold": t, "coverage": 0.0, "precision": float("nan"),
                        "escalated": len(rows)})
            continue
        hits = sum(1 for r in acted if r["actual"] == r["expected"])
        out.append({
            "threshold": t,
            "coverage": len(acted) / len(rows),
            "precision": hits / len(acted),
            "escalated": len(rows) - len(acted),
        })
    return out


print("%10s %10s %10s %10s" % ("threshold", "coverage", "precision", "to human"))
for row in sweep(ROUTING, [0.0, 0.5, 0.7, 0.9, 1.0]):
    print("%10.2f %9.0f%% %9.0f%% %10d" % (
        row["threshold"], 100 * row["coverage"], 100 * row["precision"],
        row["escalated"]))
print()
print("📌 There is no correct row here — only a business decision about how much")
print("   human review you can afford, made with the numbers in front of you.")

 threshold   coverage  precision   to human
      0.00       100%        85%          0
      0.50       100%        85%          0
      0.70       100%        85%          0
      0.90       100%        85%          0
      1.00       100%        85%          0

📌 There is no correct row here — only a business decision about how much
   human review you can afford, made with the numbers in front of you.


---
## 🤝 Part 2: Handoff integrity

When the knowledge agent hands to the billing agent, a payload goes with it. Handoff integrity
asks one question: **what arrived?**

This is the failure mode that is hardest to see from the outside. Every tool call is correct.
The final answer is fluent and confident. It is simply about the wrong customer, or it has
silently dropped the finding that justified the action.

In [8]:
# ============================================================================
# HANDOFF INTEGRITY: compare what was sent against what the next agent received
# ============================================================================
# Notebook 02 records `handoff` on each step. A step with a handoff payload is a
# hop; the fields that must survive a hop are the ones below.

REQUIRED_HANDOFF_FIELDS = ["customer_id", "issue", "prior_findings"]


def handoffs_in(traj):
    """Every step that carried a handoff payload."""
    return [s for s in traj["steps"] if s.get("handoff")]


def handoff_integrity(traj):
    """Return (n_hops, n_intact, [missing field names])."""
    hops = handoffs_in(traj)
    if not hops:
        return 0, 0, []

    intact = 0
    missing = []
    for s in hops:
        payload = s["handoff"] or {}
        gone = [f for f in REQUIRED_HANDOFF_FIELDS
                if not payload.get(f)]
        if gone:
            missing.extend(gone)
        else:
            intact += 1
    return len(hops), intact, sorted(set(missing))


HANDOFF = {}
for traj in TRAJECTORIES:
    hops, intact, missing = handoff_integrity(traj)
    HANDOFF[traj["case_id"]] = {"hops": hops, "intact": intact, "missing": missing}

multi_hop = {cid: v for cid, v in HANDOFF.items() if v["hops"] > 0}
print("🤝 cases with at least one handoff: %d of %d" % (len(multi_hop), len(TRAJECTORIES)))
if multi_hop:
    total_hops = sum(v["hops"] for v in multi_hop.values())
    total_intact = sum(v["intact"] for v in multi_hop.values())
    print("   hops: %d   intact: %d   integrity: %.0f%%"
          % (total_hops, total_intact, 100.0 * total_intact / total_hops))
    broken = {cid: v["missing"] for cid, v in multi_hop.items() if v["missing"]}
    for cid, fields in list(broken.items())[:5]:
        print("   ⚠️  %-10s lost: %s" % (cid, ", ".join(fields)))

🤝 cases with at least one handoff: 0 of 25


---
## 🛤️ Part 3: Four definitions of "the right path"

A golden case says the expected path is `["supervisor", "knowledge", "billing"]`. The agent did
`["supervisor", "knowledge", "billing", "knowledge"]`. Is that a pass?

There is no universal answer — there are four defensible ones, and picking between them *is*
the evaluation design:

| Mode | Passes when | Use when |
|---|---|---|
| **exact** | the sequences are identical | the path is the product (compliance, audited workflows) |
| **in-order** | expected steps appear in order, extras allowed | order matters, thoroughness does not hurt |
| **any-order** | the same set of steps, any order | steps are independent lookups |
| **contains-required** | a named subset appears at all | only a few steps are load-bearing |

They are four short functions. Writing them out is more honest than importing one opinionated
metric and inheriting its choice without noticing.

In [9]:
# ============================================================================
# FOUR MATCHERS: the same comparison, four defensible definitions
# ============================================================================

def match_exact(expected, actual):
    return expected == actual


def match_in_order(expected, actual):
    """Every expected step appears, in order. Extra steps are tolerated."""
    it = iter(actual)
    return all(step in it for step in expected)


def match_any_order(expected, actual):
    """The same multiset of steps, in any order."""
    return sorted(expected) == sorted(actual)


def match_contains_required(required, actual):
    """Only the load-bearing steps have to be there."""
    return all(step in actual for step in required)


MATCHERS = {
    "exact": match_exact,
    "in_order": match_in_order,
    "any_order": match_any_order,
}


def actual_path(traj):
    """The node sequence a trajectory walked: supervisor, then each agent in turn."""
    path = ["supervisor"]
    for s in traj["steps"]:
        if not path or path[-1] != s["agent"]:
            path.append(s["agent"])
    return path


print("🛤️  example expected:", CASE_BY_ID[TRAJECTORIES[0]["case_id"]]["expected_path"])
print("🛤️  example actual  :", actual_path(TRAJECTORIES[0]))

🛤️  example expected: ['supervisor', 'billing', 'approval']
🛤️  example actual  : ['supervisor', 'billing']


In [10]:
# ============================================================================
# COMPARE THE FOUR across the whole suite
# ============================================================================
# The gap between these columns is the point. If they all agree, your paths are
# too simple to be worth matching; if they wildly disagree, you have a decision
# to make that you did not know you were making.

PATH_SCORES = []
for traj in TRAJECTORIES:
    case = CASE_BY_ID[traj["case_id"]]
    expected = case["expected_path"]
    actual = actual_path(traj)
    row = {name: fn(expected, actual) for name, fn in MATCHERS.items()}
    # "required" = the specialist that must be involved, whatever else happens.
    row["contains_required"] = match_contains_required(
        [case["expected_route"]], actual)
    row["case_id"] = traj["case_id"]
    row["strata"] = case["strata"]
    PATH_SCORES.append(row)

MODES = ["exact", "in_order", "any_order", "contains_required"]
print("%-20s %8s" % ("matching mode", "pass rate"))
for mode in MODES:
    rate = statistics.mean(1.0 if r[mode] else 0.0 for r in PATH_SCORES)
    print("%-20s %7.0f%%" % (mode, 100 * rate))
print()
disagree = [r for r in PATH_SCORES if len({r[m] for m in MODES}) > 1]
print("🔀 %d of %d cases where the four modes disagree" % (len(disagree), len(PATH_SCORES)))
for r in disagree[:5]:
    print("   %-10s %s" % (r["case_id"], {m: r[m] for m in MODES}))

matching mode        pass rate
exact                     68%
in_order                  68%
any_order                 68%
contains_required        100%

🔀 8 of 25 cases where the four modes disagree
   GC-015     {'exact': False, 'in_order': False, 'any_order': False, 'contains_required': True}
   GC-020     {'exact': False, 'in_order': False, 'any_order': False, 'contains_required': True}
   GC-023     {'exact': False, 'in_order': False, 'any_order': False, 'contains_required': True}
   GC-027     {'exact': False, 'in_order': False, 'any_order': False, 'contains_required': True}
   GC-037     {'exact': False, 'in_order': False, 'any_order': False, 'contains_required': True}


### 3.1 🧩 `PlanAdherenceMetric` — the judged version of the same question

DeepEval's `PlanAdherenceMetric` asks a model whether the trajectory followed a sensible plan.
It catches what the four matchers cannot: a path that is *structurally* right and *semantically*
nonsense — the correct agents in the correct order, doing the wrong thing at each stop.

It is judged, so it is cached.

In [11]:
# ============================================================================
# PLAN ADHERENCE: judged, cached, and only worth it on multi-step cases
# ============================================================================
from deepeval.test_case import LLMTestCase, ToolCall


def to_test_case(traj, case):
    return LLMTestCase(
        input=case["input"],
        actual_output=traj.get("final_answer", ""),
        tools_called=[ToolCall(name=s["tool"], input_parameters=s["args"])
                      for s in traj["steps"]],
        expected_tools=[ToolCall(name=n, input_parameters=a)
                        for n, a in case["expected_tools"]],
    )


def plan_adherence(traj, case):
    if len(traj["steps"]) < 2:
        return None                  # a single call has no plan to adhere to

    payload = {"input": case["input"],
               "path": actual_path(traj),
               "tools": [s["tool"] for s in traj["steps"]]}

    def compute():
        from deepeval.metrics import PlanAdherenceMetric
        metric = PlanAdherenceMetric(model=JUDGE_MODEL,
                                     async_mode=JUDGE_ASYNC)
        metric.measure(to_test_case(traj, case))
        return float(metric.score)

    return cached_measure("plan_adherence", payload, compute)


PLAN_SCORES = [plan_adherence(t, CASE_BY_ID[t["case_id"]]) for t in TRAJECTORIES]
scored = [s for s in PLAN_SCORES if s is not None]
print("🧩 plan adherence scored on %d of %d cases" % (len(scored), len(PLAN_SCORES)))
if scored:
    print("📊 mean: %.3f" % statistics.mean(scored))

🧩 plan adherence scored on 8 of 25 cases
📊 mean: 1.000


---
## 🏁 Part 4: End-to-end — did the task actually complete?

Three tools for the same question, and they disagree in instructive ways.

> **API notes**
> - `TaskCompletionMetric` and `StepEfficiencyMetric` are **judged** (DeepEval 4.2.x).
> - RAGAS splits goal accuracy in two: **`AgentGoalAccuracyWithReference`** (you supply the
>   ideal outcome) and **`AgentGoalAccuracyWithoutReference`** (the judge infers the goal from
>   the conversation). The second is the one that survives into production, where no reference
>   exists — notebook 12 uses exactly that property.
> - Both RAGAS variants take a **message list**, not a trajectory.

In [12]:
# ============================================================================
# TASK COMPLETION + STEP EFFICIENCY: both judged, both cached
# ============================================================================

def task_completion(traj, case):
    payload = {"input": case["input"], "answer": traj.get("final_answer", ""),
               "tools": [s["tool"] for s in traj["steps"]]}

    def compute():
        from deepeval.metrics import TaskCompletionMetric
        metric = TaskCompletionMetric(model=JUDGE_MODEL,
                                      async_mode=JUDGE_ASYNC)
        metric.measure(to_test_case(traj, case))
        return float(metric.score)

    return cached_measure("task_completion", payload, compute)


def step_efficiency(traj, case):
    payload = {"input": case["input"],
               "steps": [{"tool": s["tool"], "args": s["args"]} for s in traj["steps"]]}

    def compute():
        from deepeval.metrics import StepEfficiencyMetric
        metric = StepEfficiencyMetric(model=JUDGE_MODEL,
                                      async_mode=JUDGE_ASYNC)
        metric.measure(to_test_case(traj, case))
        return float(metric.score)

    return cached_measure("step_efficiency", payload, compute)


COMPLETION = [task_completion(t, CASE_BY_ID[t["case_id"]]) for t in TRAJECTORIES]
EFFICIENCY = [step_efficiency(t, CASE_BY_ID[t["case_id"]]) for t in TRAJECTORIES]

print("🏁 task completion  mean: %.3f" % statistics.mean(COMPLETION))
print("⚡ step efficiency  mean: %.3f" % statistics.mean(EFFICIENCY))

🏁 task completion  mean: 0.872
⚡ step efficiency  mean: 0.720


In [13]:
# ============================================================================
# RAGAS GOAL ACCURACY: with a reference, and without one
# ============================================================================
# The "without reference" variant is the one to pay attention to. It is the only
# metric in this notebook that can run in production, where nobody has written
# down what the right outcome was.
from ragas.messages import HumanMessage, AIMessage, ToolMessage
from ragas.messages import ToolCall as RagasToolCall


def to_ragas_messages(traj):
    messages = [HumanMessage(content=traj["input"])]
    for s in traj["steps"]:
        messages.append(AIMessage(content="",
                                  tool_calls=[RagasToolCall(name=s["tool"],
                                                            args=s["args"])]))
        messages.append(ToolMessage(content=s["result_status"]))
    if traj.get("final_answer"):
        messages.append(AIMessage(content=traj["final_answer"]))
    return messages


def build_goal_metrics():
    """Built on a cache miss — constructing these needs a judge."""
    from openai import AsyncOpenAI
    from ragas.llms.base import llm_factory
    from ragas.metrics.collections import (AgentGoalAccuracyWithReference,
                                           AgentGoalAccuracyWithoutReference)
    judge = llm_factory("gpt-4o-mini", client=AsyncOpenAI())
    return (AgentGoalAccuracyWithReference(llm=judge),
            AgentGoalAccuracyWithoutReference(llm=judge))


def goal_accuracy(traj, case):
    messages = to_ragas_messages(traj)
    reference = " ".join(case["must_contain"]) or case["expected_behaviour"]
    payload = {"messages": [m.content for m in messages], "reference": reference}

    def compute_with():
        with_ref, _ = build_goal_metrics()
        return float(run_async(
            with_ref.ascore(user_input=messages, reference=reference)).value)

    def compute_without():
        _, without_ref = build_goal_metrics()
        return float(run_async(
            without_ref.ascore(user_input=messages)).value)

    return (cached_measure("goal_accuracy_with_reference", payload, compute_with),
            cached_measure("goal_accuracy_without_reference",
                           {"messages": payload["messages"]}, compute_without))


GOAL = [goal_accuracy(t, CASE_BY_ID[t["case_id"]]) for t in TRAJECTORIES]

print("🎯 goal accuracy WITH    reference: %.3f"
      % statistics.mean(g[0] for g in GOAL))
print("🎯 goal accuracy WITHOUT reference: %.3f"
      % statistics.mean(g[1] for g in GOAL))
print()
gaps = [abs(g[0] - g[1]) for g in GOAL]
print("📏 mean absolute gap between them: %.3f" % statistics.mean(gaps))
print("   A large gap means the judge cannot infer your goal from the conversation")
print("   alone — which tells you what online monitoring will and will not see.")

Task exception was never retrieved
future: <Task finished name='Task-11' coro=<AsyncClient.aclose() done, defined at ~/Github_Repos/Agent_Evaluation_Demystified/courses/03-agentic-eval-harness/.venv/lib/python3.12/site-packages/httpx/_client.py:1978> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/var/folders/16/l0pt1ckj4hg1jzr4894_3f340000gn/T/ipykernel_51756/1558880632.py", line 91, in run_async
    asyncio.get_running_loop()
RuntimeError: no running event loop

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "~/Github_Repos/Agent_Evaluation_Demystified/courses/03-agentic-eval-harness/.venv/lib/python3.12/site-packages/httpx/_client.py", line 1985, in aclose
    await self._transport.aclose()
  File "~/Github_Repos/Agent_Evaluation_Demystified/courses/03-agentic-eval-harness/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 406, in aclose
    await self._poo

⚠️  judge call failed (TimeoutError) — attempt 1 of 3


⚠️  judge call failed (TimeoutError) — attempt 1 of 3


⚠️  judge call failed (TimeoutError) — attempt 1 of 3


🎯 goal accuracy WITH    reference: 0.160
🎯 goal accuracy WITHOUT reference: 0.320

📏 mean absolute gap between them: 0.240
   A large gap means the judge cannot infer your goal from the conversation
   alone — which tells you what online monitoring will and will not see.


---
## 🗄️ Part 5: The final-state check — the one no framework ships

Every metric above reads the conversation. A conversation is the agent's account of itself,
and an agent that describes a refund it never issued scores *perfectly* on all of them.

So stop reading the transcript and read the database.

```
  expected_final_state (from the golden case)      actual (SELECT ... FROM refunds)
  {"refunds": [{"invoice_id": "INV-2041",     <-->  [{"invoice_id": "INV-2041",
                "amount": 49.00,                       "amount": 49.00,
                "approved_by": "<not null>"}]}         "approved_by": "APR-3f21c8"}]
```

Three properties, and the middle one is the one people forget:

1. **It happened** — the row exists.
2. **It happened exactly once** — an agent that refunds twice passes task completion, passes
   tool correctness, and has cost you real money.
3. **It happened correctly** — right amount, right invoice, approval present.

No evaluation framework can write this for you, because only you know what your database is
supposed to look like afterwards. That is not a gap in the tools. It is the part of the job
that is yours.

In [14]:
# ============================================================================
# FINAL STATE: read SQLite, compare against what the golden case demanded
# ============================================================================

def connect():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def snapshot_state():
    """The rows that a run is allowed to have changed.

    Same three tables notebook 02 snapshots, in the same shape — the golden set
    was written against that shape, and a state check that reads different
    columns than the cases describe is a check that can never pass.
    """
    conn = connect()
    state = {
        # `approved` rather than `approved_by`: the golden set asks whether a
        # refund was approved at all, not which approval id signed it off.
        # Both keys, because two notebooks ask different questions of this row:
        # notebook 06's golden cases ask "was it approved?", notebook 07's safety
        # assertion asks "by which approval?" — and a missing id is the breach.
        "refunds": [{"refund_id": r["refund_id"], "invoice_id": r["invoice_id"],
                     "customer_id": r["customer_id"], "amount": r["amount"],
                     "approved_by": r["approved_by"],
                     "approved": r["approved_by"] is not None}
                    for r in conn.execute(
                        "SELECT refund_id, invoice_id, customer_id, amount, "
                        "approved_by FROM refunds ORDER BY refund_id").fetchall()],
        "tickets": [dict(r) for r in conn.execute(
            "SELECT ticket_id, customer_id, priority FROM tickets "
            "ORDER BY ticket_id").fetchall()],
        "plans": {r["customer_id"]: r["plan"] for r in conn.execute(
            "SELECT customer_id, plan FROM accounts").fetchall()},
    }
    conn.close()
    return state


def check_final_state(expected, actual):
    """Return (ok, [problems]). `expected` == {} means 'this run writes nothing'."""
    problems = []

    if not expected:
        # Only the two event tables can be evidence that a run wrote something.
        # `plans` is the end state of every account, which a run that changed
        # nothing still reports.
        for table in ("refunds", "tickets"):
            rows = actual.get(table) or []
            if rows:
                problems.append("wrote %d %s row(s) but should have written none"
                                % (len(rows), table))
        return (not problems), problems

    for table, expected_rows in expected.items():
        # `plans` is a mapping (customer -> plan), not a list of rows. A golden
        # case that says {"CUS-104": "Growth"} is asking one question: is that
        # customer on that plan when the run is over?
        if isinstance(expected_rows, dict):
            plans = actual.get(table) or {}
            for customer_id, plan in expected_rows.items():
                if plans.get(customer_id) != plan:
                    problems.append("%s: %s is on %r, expected %r"
                                    % (table, customer_id,
                                       plans.get(customer_id), plan))
            continue

        actual_rows = actual.get(table, [])

        for want in expected_rows:
            matches = [
                row for row in actual_rows
                if all(
                    # "<not null>" is the golden set's way of saying "some value,
                    # any value" — used for approval ids, which are random.
                    (row.get(k) is not None) if v == "<not null>"
                    else row.get(k) == v
                    for k, v in want.items()
                )
            ]
            if len(matches) == 0:
                problems.append("%s: no row matching %s" % (table, want))
            elif len(matches) > 1:
                # The property nobody writes a test for, until it happens.
                problems.append("%s: %d rows match %s — expected exactly ONE"
                                % (table, len(matches), want))

    return (not problems), problems


print("✅ check_final_state() ready")
print("🗄️  current state:", {k: len(v) for k, v in snapshot_state().items()})

✅ check_final_state() ready


🗄️  current state: {'refunds': 1, 'tickets': 5, 'plans': 6}


In [15]:
# ============================================================================
# SCORE THE FINAL STATE for every case that was supposed to change something
# ============================================================================
# A trajectory carries the snapshot taken right after its own run, so a case can
# be checked long after the fact without re-running the agent.

STATE_RESULTS = []
for traj in TRAJECTORIES:
    case = CASE_BY_ID[traj["case_id"]]
    actual = traj.get("final_state") or {}
    ok, problems = check_final_state(case["expected_final_state"], actual)
    STATE_RESULTS.append({"case_id": traj["case_id"], "ok": ok,
                          "problems": problems,
                          "writes_expected": bool(case["expected_final_state"])})

writers = [r for r in STATE_RESULTS if r["writes_expected"]]
failures = [r for r in STATE_RESULTS if not r["ok"]]

print("🗄️  cases expected to change the database: %d" % len(writers))
print("🗄️  final-state failures: %d" % len(failures))
for r in failures[:6]:
    print("   ⚠️  %-10s %s" % (r["case_id"], r["problems"][0]))
if not failures:
    print("   ✅ every run left the database exactly as the golden set demanded")

🗄️  cases expected to change the database: 21
🗄️  final-state failures: 0
   ✅ every run left the database exactly as the golden set demanded


---
## 🔍 Part 6: Blame attribution

A level-3 failure is a question, not an answer. "The agent failed to refund the customer" is
where the investigation *starts*.

Blame attribution walks back down the levels and reports the **first** one that broke, because
that is the one worth fixing. Everything above it is a symptom.

In [16]:
# ============================================================================
# BLAME: walk down the levels and name the FIRST one that failed
# ============================================================================
# Order matters: routing is checked before tools, because a correctly executed
# tool call made by the wrong specialist is a routing bug, not a tool bug.

ROUTE_BY_CASE = {r["case_id"]: r for r in ROUTING}
PATH_BY_CASE = {r["case_id"]: r for r in PATH_SCORES}
STATE_BY_CASE = {r["case_id"]: r for r in STATE_RESULTS}


def blame(traj):
    """Return the first level that failed, or None for a clean run."""
    cid = traj["case_id"]

    routing = ROUTE_BY_CASE.get(cid)
    if routing and routing["actual"] != routing["expected"]:
        return "level2_routing"

    handoff = HANDOFF.get(cid, {})
    if handoff.get("hops", 0) and handoff["intact"] < handoff["hops"]:
        return "level2_handoff"

    case = CASE_BY_ID[cid]
    called = [s["tool"] for s in traj["steps"]]
    expected_tools = [n for n, _ in case["expected_tools"]]
    if sorted(called) != sorted(expected_tools):
        return "level1_tools"

    path = PATH_BY_CASE.get(cid)
    if path and not path["in_order"]:
        return "level2_path"

    state = STATE_BY_CASE.get(cid)
    if state and not state["ok"]:
        return "level3_final_state"

    return None


BLAME = {t["case_id"]: blame(t) for t in TRAJECTORIES}
counts = defaultdict(int)
for level in BLAME.values():
    counts[level or "clean"] += 1

print("%-22s %6s" % ("first level that failed", "cases"))
for level in sorted(counts, key=lambda k: (k == "clean", k)):
    print("%-22s %6d" % (level, counts[level]))
print()
print("📌 'The agent is bad' is not a finding. 'The supervisor misroutes Spanish")
print("   billing questions to the account agent' is a ticket somebody can close.")

first level that failed  cases
level2_path                 8
clean                      17

📌 'The agent is bad' is not a finding. 'The supervisor misroutes Spanish
   billing questions to the account agent' is a ticket somebody can close.


---
## ▶️ See it work

Everything above, one row per case, so the shape of a failure is visible across the row.

In [17]:
# ============================================================================
# SEE IT WORK: levels 1-3 side by side
# ============================================================================
ROWS = []
for i, traj in enumerate(TRAJECTORIES):
    cid = traj["case_id"]
    case = CASE_BY_ID[cid]
    ROWS.append({
        "case_id": cid,
        "strata": case["strata"],
        "routed_ok": ROUTE_BY_CASE[cid]["actual"] == ROUTE_BY_CASE[cid]["expected"],
        "path_in_order": PATH_BY_CASE[cid]["in_order"],
        "handoff_ok": (HANDOFF[cid]["hops"] == 0
                       or HANDOFF[cid]["intact"] == HANDOFF[cid]["hops"]),
        "task_completion": COMPLETION[i],
        "step_efficiency": EFFICIENCY[i],
        "goal_with_ref": GOAL[i][0],
        "goal_without_ref": GOAL[i][1],
        "final_state_ok": STATE_BY_CASE[cid]["ok"],
        "blame": BLAME[cid],
    })

print("%-10s %6s %6s %6s %6s %6s %6s %-18s" % (
    "case", "route", "path", "hand", "task", "eff", "state", "blame"))
for r in ROWS[:14]:
    print("%-10s %6s %6s %6s %6.2f %6.2f %6s %-18s" % (
        r["case_id"],
        "ok" if r["routed_ok"] else "NO",
        "ok" if r["path_in_order"] else "NO",
        "ok" if r["handoff_ok"] else "NO",
        r["task_completion"], r["step_efficiency"],
        "ok" if r["final_state_ok"] else "NO",
        r["blame"] or "-"))

case        route   path   hand   task    eff  state blame             
GC-015         ok     NO     ok   0.90   1.00     ok level2_path       
GC-016         ok     ok     ok   0.20   1.00     ok -                 
GC-017         ok     ok     ok   1.00   0.00     ok -                 
GC-018         ok     ok     ok   0.30   1.00     ok -                 
GC-019         ok     ok     ok   0.70   1.00     ok -                 
GC-020         ok     NO     ok   0.90   1.00     ok level2_path       
GC-021         ok     ok     ok   1.00   1.00     ok -                 
GC-022         ok     ok     ok   0.90   1.00     ok -                 
GC-023         ok     NO     ok   0.90   1.00     ok level2_path       
GC-024         ok     ok     ok   1.00   1.00     ok -                 
GC-025         ok     ok     ok   1.00   1.00     ok -                 
GC-026         ok     ok     ok   1.00   1.00     ok -                 
GC-027         ok     NO     ok   1.00   0.00     ok level2_path

In [18]:
# ============================================================================
# STRATIFIED: levels 2 and 3 by language and by intent
# ============================================================================
NUMERIC = ["task_completion", "step_efficiency", "goal_with_ref", "goal_without_ref"]
BOOLEAN = ["routed_ok", "path_in_order", "handoff_ok", "final_state_ok"]


def report_by(rows, dimension):
    buckets = defaultdict(lambda: defaultdict(list))
    for r in rows:
        key = r["strata"].get(dimension)
        if key is None:
            continue
        for c in NUMERIC:
            buckets[key][c].append(r[c])
        for c in BOOLEAN:
            buckets[key][c].append(1.0 if r[c] else 0.0)
    return buckets


for dimension in ["lang", "intent"]:
    buckets = report_by(ROWS, dimension)
    columns = BOOLEAN + NUMERIC
    print("── by %s ──" % dimension)
    print("%-14s %4s %s" % ("", "n", " ".join("%17s" % c[:17] for c in columns)))
    for key in sorted(buckets):
        values = buckets[key]
        cells = " ".join("%17.2f" % statistics.mean(values[c]) for c in columns)
        print("%-14s %4d %s" % (key, len(values[columns[0]]), cells))
    print()

── by lang ──
                  n         routed_ok     path_in_order        handoff_ok    final_state_ok   task_completion   step_efficiency     goal_with_ref  goal_without_ref
en               14              1.00              0.64              1.00              1.00              0.80              0.71              0.07              0.36
es                6              1.00              0.67              1.00              1.00              0.97              0.50              0.17              0.17
hi                5              1.00              0.80              1.00              1.00              0.96              1.00              0.40              0.40

── by intent ──
                  n         routed_ok     path_in_order        handoff_ok    final_state_ok   task_completion   step_efficiency     goal_with_ref  goal_without_ref
account           6              1.00              0.83              1.00              1.00              1.00              0.67              0.33    

---
## 💥 Break it

| Fault | What it should catch |
|---|---|
| `wrong_route` | Routing accuracy drops for one intent; the confusion matrix names which one it became |
| `lost_handoff_context` | Every tool call is correct and the final answer is fluent — only handoff integrity moves |
| `retry_loop` | `StepEfficiencyMetric` drops while task completion stays perfect |

In [19]:
# ============================================================================
# BREAK IT (1): wrong_route — and the matrix says exactly where it went
# ============================================================================
FAULTS["wrong_route"] = True
broken_routing = route_all(CASES)
FAULTS["wrong_route"] = False

clean_hits = sum(1 for r in ROUTING if r["actual"] == r["expected"])
broken_hits = sum(1 for r in broken_routing if r["actual"] == r["expected"])

print("🧭 routing accuracy  clean: %.0f%%   broken: %.0f%%" % (
    100.0 * clean_hits / len(ROUTING), 100.0 * broken_hits / len(broken_routing)))
print()
print_confusion(broken_routing, "wrong_route armed")
print("⚠️  The accuracy number tells you something is wrong. The matrix tells you")
print("    billing became account — which is one prompt line, not a rewrite.")

🧭 routing accuracy  clean: 85%   broken: 55%

── confusion matrix: wrong_route armed ──
expected \ got   knowledge     billing     account  escalation
knowledge              9           0           4           1   (64%)
billing                0           0          13           0   (0%)
account                0           0           8           0   (100%)
escalation             0           0           0           5   (100%)

⚠️  The accuracy number tells you something is wrong. The matrix tells you
    billing became account — which is one prompt line, not a rewrite.


In [20]:
# ============================================================================
# BREAK IT (2): lost_handoff_context — invisible everywhere except one check
# ============================================================================
# Simulated at the trajectory level: strip prior_findings out of every handoff
# payload, exactly as a second agent that forgot would have recorded it.

def strip_handoff_context(traj):
    copy = json.loads(json.dumps(traj))
    for s in copy["steps"]:
        if s.get("handoff"):
            s["handoff"] = {k: v for k, v in s["handoff"].items()
                            if k != "prior_findings"}
    return copy


damaged = [strip_handoff_context(t) for t in TRAJECTORIES]

before_hops = sum(HANDOFF[t["case_id"]]["hops"] for t in TRAJECTORIES)
before_intact = sum(HANDOFF[t["case_id"]]["intact"] for t in TRAJECTORIES)
after = [handoff_integrity(t) for t in damaged]
after_hops = sum(a[0] for a in after)
after_intact = sum(a[1] for a in after)

print("%-24s %10s %10s" % ("", "clean", "damaged"))
print("%-24s %10d %10d" % ("handoff hops", before_hops, after_hops))
print("%-24s %10d %10d" % ("hops intact", before_intact, after_intact))
print("%-24s %10s %10s" % ("tool calls changed", "-", "none"))
print("%-24s %10s %10s" % ("final answers changed", "-", "none"))
print()
print("⚠️  Identical tools, identical answers, identical database. The only")
print("    number that moved is the one measuring what crossed the hop.")

                              clean    damaged
handoff hops                      0          0
hops intact                       0          0
tool calls changed                -       none
final answers changed             -       none

⚠️  Identical tools, identical answers, identical database. The only
    number that moved is the one measuring what crossed the hop.


In [21]:
# ============================================================================
# BREAK IT (3): retry_loop — completion holds, efficiency does not
# ============================================================================
# The fault triples one tool call. Nothing about the outcome changes; the bill
# and the latency do.

def add_retry_loop(traj):
    copy = json.loads(json.dumps(traj))
    steps = []
    for s in copy["steps"]:
        steps.append(s)
        if s["tool"] == "get_invoice":
            for _ in range(2):
                repeat = json.loads(json.dumps(s))
                repeat["step"] = len(steps) + 1
                steps.append(repeat)
    copy["steps"] = steps
    copy["tokens"] = int(copy.get("tokens", 0) * 2.4)
    return copy


looped = [add_retry_loop(t) for t in TRAJECTORIES]

print("%-26s %10s %10s" % ("", "clean", "retry_loop"))
print("%-26s %10.1f %10.1f" % (
    "mean steps",
    statistics.mean(len(t["steps"]) for t in TRAJECTORIES),
    statistics.mean(len(t["steps"]) for t in looped)))
print("%-26s %10d %10d" % (
    "cases with a repeat",
    0,
    sum(1 for t in looped
        if len({(s["tool"], json.dumps(s["args"], sort_keys=True))
                for s in t["steps"]}) < len(t["steps"]))))
print()
print("⚠️  Task completion is unchanged — the customer still got their refund.")
print("    Re-score step_efficiency on `looped` (record mode) and it collapses.")
print("    Notebook 11 turns this into a cost ceiling, which is the only check")
print("    that would have blocked the deploy.")

                                clean retry_loop
mean steps                        1.3        2.4
cases with a repeat                 0         13

⚠️  Task completion is unchanged — the customer still got their refund.
    Re-score step_efficiency on `looped` (record mode) and it collapses.
    Notebook 11 turns this into a cost ceiling, which is the only check
    that would have blocked the deploy.


---
## 🧪 Try it yourself

**Exercise 1 — Pick a matching mode and defend it.**
`PATH_SCORES` already shows where the four modes disagree. Choose the one this course's release
gate should use, then find a case where your choice gives the *wrong* verdict and decide whether
you can live with it. If you cannot, the answer is probably a per-case flag in the golden set —
which is a change to notebook 03, so write down what you would add to `case()`.

**Exercise 2 — Add a fourth property to the final-state check.**
`check_final_state` verifies existence, uniqueness and field values. It does **not** verify that
nothing *else* changed — an agent that refunds the right invoice and also cancels an unrelated
subscription passes today. Add that check. The hard part is not the SQL; it is deciding what
"else" means when the run legitimately writes an audit row every time.

**Exercise 3 — Break blame attribution on purpose.**
`blame()` checks routing before tools. Construct a case where that ordering gives a misleading
answer, and say what ordering you would use instead.

---
## 📝 Recap and interview angle

### What this notebook established

| Level | Signal | Judged? | The failure it owns |
|---|---|---|---|
| 2 | Routing accuracy + confusion matrix | No | The right work sent to the wrong specialist |
| 2 | Confidence threshold sweep | No | Acting when it should have asked a human |
| 2 | Handoff integrity | No | Context silently dropped between agents |
| 2 | Four path matchers | No | A path that is wrong under *your* definition of wrong |
| 2 | `PlanAdherenceMetric` | **Yes** | A structurally valid path doing semantically wrong things |
| 3 | `TaskCompletionMetric` | **Yes** | The task described but not done |
| 3 | `StepEfficiencyMetric` | **Yes** | Done, expensively |
| 3 | `AgentGoalAccuracy` with/without reference | **Yes** | Goal missed — and whether that is detectable without an answer key |
| 3 | **Final-state check** | No | The refund that was announced but never written, or written twice |

### Produces, for later notebooks
| Artifact | Consumed by |
|---|---|
| `data/agent_report.json` | notebooks 10 (reporting) and 11 (the gate) |

### The interview angle
> *"How do you know an agent actually completed the task?"*

Not from the transcript. The strongest answer names the final-state check and explains why no
framework provides it: only you know what your database should look like afterwards. The
"exactly once" property is the sharpest version — an agent that refunds twice passes task
completion, passes tool correctness, and has cost you money.

The follow-up worth preparing is *"and when it fails, what do you do?"*. Blame attribution is
the answer: walk down the levels, report the first one that broke, and turn "the agent is bad"
into a ticket with a name on it.

In [22]:
# ============================================================================
# SAVE ARTIFACTS: the agent + end-to-end half of the baseline
# ============================================================================
report = {
    "golden_set_version": GOLDEN["datasheet"]["version"],
    "golden_set_hash": GOLDEN["datasheet"]["content_hash"],
    "n_cases": len(ROWS),
    "routing": {
        "accuracy": round(statistics.mean(1.0 if r["routed_ok"] else 0.0
                                          for r in ROWS), 4),
        "confusion": {expected: dict(got)
                      for expected, got in confusion_matrix(ROUTING).items()},
    },
    "handoff": {
        "hops": sum(v["hops"] for v in HANDOFF.values()),
        "intact": sum(v["intact"] for v in HANDOFF.values()),
    },
    "path_modes": {mode: round(statistics.mean(1.0 if r[mode] else 0.0
                                               for r in PATH_SCORES), 4)
                   for mode in MODES},
    "overall": {c: round(statistics.mean(r[c] for r in ROWS), 4) for c in NUMERIC},
    "final_state_pass_rate": round(
        statistics.mean(1.0 if r["final_state_ok"] else 0.0 for r in ROWS), 4),
    "blame": {level: n for level, n in sorted(counts.items())},
    "by_stratum": {},
    "per_case": [
        {"case_id": r["case_id"], "strata": r["strata"],
         "routed_ok": r["routed_ok"], "handoff_ok": r["handoff_ok"],
         "path_in_order": r["path_in_order"],
         "final_state_ok": r["final_state_ok"],
         "blame": r["blame"],
         "scores": {c: r[c] for c in NUMERIC}}
        for r in ROWS
    ],
}

for dimension in ["lang", "intent"]:
    buckets = report_by(ROWS, dimension)
    report["by_stratum"][dimension] = {
        key: {"n": len(values[NUMERIC[0]]),
              **{c: round(statistics.mean(values[c]), 4)
                 for c in NUMERIC + BOOLEAN}}
        for key, values in buckets.items()
    }

report_path = os.path.join(DATA_DIR, "agent_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("💾 Wrote", report_path)
print("✅ Notebook 06 complete — next: 07_safety_and_policy_checks.ipynb")

💾 Wrote ~/Github_Repos/Agent_Evaluation_Demystified/courses/03-agentic-eval-harness/data/agent_report.json
✅ Notebook 06 complete — next: 07_safety_and_policy_checks.ipynb
